# Projekt mastermind

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Projekt: Mastermind</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 13: Abschlussprojekt &nbsp;|&nbsp; ca. 2 x 90 Min.</p>
</div>
</div>

> **[Kursinhalt]** Dieses Notebook enthaelt keinen fertigen Code. Es beschreibt das Projekt und gibt Hinweise -- die Implementierung liegt bei dir.

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Was soll entstehen?</span>
</div>

Mastermind ist ein Logikspiel fuer zwei Personen -- in unserer Version spielt der Mensch gegen den Computer.

Das Spiel funktioniert so: Der Computer denkt sich zu Beginn eine geheime Farbkombination aus -- zum Beispiel vier Farben aus einer Palette von sechs, mit Wiederholungen erlaubt oder nicht. Der Spieler hat eine begrenzte Anzahl Versuche um die Kombination zu erraten. Nach jedem Versuch gibt der Computer Hinweise:

- **Schwarz** (oder ein anderes Signal): Eine Farbe ist richtig und steht an der richtigen Position
- **Weiss** (oder ein anderes Signal): Eine Farbe ist enthalten, steht aber an der falschen Position

Der Spieler nutzt diese Hinweise um seinen naechsten Versuch einzugrenzen. Das Spiel endet wenn der Spieler die Kombination erraten hat -- oder wenn die maximale Anzahl Versuche ausgeschoepft ist.

Klingt einfach. Ist es nicht. Die Logik fuer die Hinweise ist tricky, die Randfaelle sind zahlreich, und wenn man eine GUI baut kommt nochmal eine ganze Schicht Komplexitaet dazu.

Wie dein Mastermind aussieht entscheidest du. Eine Konsolenversion ist ein vollwertiges Programm. Eine grafische Version ist anspruchsvoller aber beeindruckender. Beides ist ein guter Abschluss des Kurses.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Warum dieses Projekt?</span>
</div>

Mastermind ist kleiner als das MP3-Projekt -- aber es ist kein einfaches Projekt. Es testet etwas anderes: reines algorithmisches Denken.

**Logik und Algorithmen** -- die Kernaufgabe ist die Bewertungsfunktion: Gegeben eine geheime Kombination und einen Versuch, berechne die Anzahl der schwarzen und weissen Treffer. Das klingt trivial -- aber wenn man es falsch implementiert, merkt man es erst wenn das Spiel seltsame Ergebnisse liefert. Das zu debuggen ist lehrreich.

**Zustandsverwaltung** -- ein laufendes Spiel hat Zustand: die geheime Kombination, alle bisherigen Versuche, die Bewertungen dazu, die Anzahl verbleibender Versuche. Wo lebt dieser Zustand? Wie kommt er von Runde zu Runde?

**Trennung von Logik und Darstellung** -- die Spiellogik (was ist ein gueltiger Versuch, wie wird er bewertet) ist unabhaengig davon ob das Spiel in der Konsole oder mit GUI laeuft. Wenn man das sauber trennt, kann man dieselbe Logik fuer beide Versionen nutzen.

**Eingabevalidierung** -- der Spieler kann falsch eingeben: zu viele Farben, ungueltige Farbbezeichnungen, Leerzeilen. Das Programm muss das abfangen und sinnvoll darauf reagieren.

**Zufaelligkeit und Reproduzierbarkeit** -- der Computer muss eine zufaellige Kombination auswaehlen. Aber wie testet man zufaelligen Code? `random.seed()` aus dem Kurs bekommt hier eine echte Anwendung.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Die Spielregeln -- genau lesen</span>
</div>

Die Standardvariante:

- 6 moegliche Farben: Rot, Gruen, Blau, Gelb, Orange, Lila
- Geheimcode: 4 Farben, Wiederholungen erlaubt
- Maximale Versuche: 10
- Hinweise pro Versuch: X schwarze Pins (richtige Farbe, richtige Position) + Y weisse Pins (richtige Farbe, falsche Position)

Die Regeln fuer die Hinweisberechnung sind die eigentliche Herausforderung. Ein Beispiel:

```
Geheimcode: ROT BLAU ROT GRUEN
Versuch:    ROT ROT  BLAU GELB

Schwarz: 1  (ROT an Position 1 ist korrekt)
Weiss:   1  (ROT kommt im Code vor, aber nicht an Position 2 --
              allerdings wurde der zweite ROT im Code schon verbraucht)
```

Das Wort "verbraucht" ist der Kern des Problems. Jede Farbe im Code kann nur einmal als Hinweis zaehlen -- egal ob schwarz oder weiss. Wenn der Code zweimal ROT hat und der Versuch dreimal ROT, dann sind maximal zwei davon Treffer.

Dieser Mechanismus fuehrt zu vielen Implementierungsfehlern. Implementiere ihn zuerst, teste ihn gruendlich, bevor du den Rest baust.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Anforderungen</span>
</div>

**Muss vorhanden sein:**

- Der Computer waehlt zufaellig einen Geheimcode
- Der Spieler gibt Versuche ein
- Nach jedem Versuch werden schwarze und weisse Treffer angezeigt
- Das Spiel endet korrekt -- Gewinn oder Niederlage
- Alle bisherigen Versuche und Bewertungen werden angezeigt
- Ungueltige Eingaben werden abgefangen

**Kann vorhanden sein (optional):**

- Schwierigkeitsgrade: mehr Farben, mehr Positionen, weniger Versuche
- Statistiken: gewonnene Spiele, durchschnittliche Versuche
- Einen Hinweis-Modus der einen moeglichen naechsten Zug vorschlaegt
- Mehrere Runden mit Punktestand
- Eine grafische Oberflaeche mit farbigen Kreisen
- Computer spielt gegen sich selbst -- loest das Spiel automatisch

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Architekturfragen -- bevor du anfaengst</span>
</div>

**Was repraesentiert eine Farbe in deinem Code?**
Ein String wie `'rot'`? Ein Integer wie `0, 1, 2, 3`? Ein Enum? Jede Wahl hat Konsequenzen fuer die Eingabe, den Vergleich und die Ausgabe. Strings sind lesbar aber anfaellig fuer Tippfehler. Integer sind schnell aber bedeutungslos ohne Mapping.

**Wie repraesentierst du den Geheimcode und einen Versuch?**
Eine Liste von vier Elementen liegt nahe. Aber wo wird sichergestellt dass es genau vier sind -- und nur gueltigen Farben? In der Klasse? Beim Einlesen der Eingabe? Beides?

**Wo lebt die Bewertungslogik?**
Die Funktion die aus Code und Versuch die schwarzen und weissen Treffer berechnet ist das Herzstuck des Spiels. Soll sie eine Methode des Spiel-Objekts sein? Eine Standalone-Funktion? Bedenke: eine Standalone-Funktion ist einfacher zu testen -- du kannst sie direkt aufrufen ohne ein ganzes Spiel aufzusetzen.

**Wie trennst du Logik und Darstellung?**
Wenn deine Spiellogik direkt `print()` aufruft, kannst du sie nicht wiederverwenden ohne Ausgabe zu produzieren. Was wenn du dieselbe Logik fuer eine GUI nutzen willst? Eine saubere Trennung bedeutet: die Logik gibt Werte zurueck, die Darstellung entscheidet wie sie angezeigt werden.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Technische Hinweise</span>
</div>

**Die Bewertungslogik -- der schwierigste Teil**

Die Berechnung der Treffer ist fehleranfaellig. Ein Ansatz der funktioniert:

```
# Pseudocode fuer die Bewertung
funktion bewerte(geheimcode, versuch):

    # Schritt 1: Schwarze Treffer -- exakte Treffer
    schwarz = 0
    fuer jede Position:
        wenn versuch[pos] == geheimcode[pos]:
            schwarz += 1
            markiere diese Position in beiden als 'verbraucht'

    # Schritt 2: Weisse Treffer -- nur auf nicht-verbrauchten Positionen
    weiss = 0
    fuer jede nicht-verbrauchte Farbe im versuch:
        wenn diese Farbe noch im nicht-verbrauchten geheimcode vorkommt:
            weiss += 1
            markiere als verbraucht

    return schwarz, weiss
```

Das Wort "verbraucht" ist entscheidend. Eine Moeglichkeit: Kopien beider Listen erstellen und erkannte Treffer durch `None` ersetzen.

---

**Zufaelligen Code erzeugen: `random`**

```
# Pseudocode
farben = ['rot', 'blau', 'gruen', 'gelb', 'orange', 'lila']
geheimcode = waehle 4 zufaellige Farben aus farben (mit Wiederholung)
```

`random.choices()` oder `random.choice()` in einer Schleife -- beide loesen das.

---

**Eingabe parsen**

Der Spieler gibt etwas ein -- aber was genau? Ein String wie `rot blau rot gruen`? Abkuerzungen wie `r b r g`? Zahlen? Das entscheidest du -- aber dokumentiere es dem Spieler.

```
# Pseudocode fuer Eingabe
eingabe = lese vom Spieler
teile eingabe bei Leerzeichen auf -> liste
wenn laenge != 4: Fehler
fuer jedes element: wenn nicht in gueltigen Farben: Fehler
gib liste zurueck
```

---

**Spielverlauf anzeigen**

Nach jedem Versuch soll der Spieler alle bisherigen Versuche und Bewertungen sehen. Eine Liste von Tupeln -- `(versuch, schwarz, weiss)` -- speichert die gesamte Historie und kann jederzeit ausgegeben werden.

---

**GUI: `tkinter`**

Wenn du eine grafische Oberflaeche willst, bietet sich `tkinter` an. Farbige Kreise lassen sich mit `Canvas`-Objekten zeichnen. `canvas.create_oval()` zeichnet einen Kreis, `fill='red'` fuellt ihn mit Farbe. Die Spiellogik bleibt dieselbe -- nur die Darstellung aendert sich.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Typische Fallen</span>
</div>

**Die Bewertungslogik bei Wiederholungen**
Der haeufigste Bug: Farben werden doppelt gezaehlt. Wenn der Code `ROT ROT BLAU GRUEN` ist und der Versuch `ROT ROT ROT ROT`, dann sind zwei Treffer schwarz und nicht vier. Der Verbrauchsmechanismus muss korrekt implementiert sein -- teste ihn mit Wiederholungen bevor du weitermachst.

**Schwarze vor weissen Treffern**
Schwarze Treffer muessen vor weissen berechnet werden. Wenn eine Farbe an der richtigen Position ist (schwarz), darf sie nicht nochmals als weisser Treffer zaehlen. Die Reihenfolge der Berechnung ist nicht optional.

**Spielzustand zwischen Runden**
Wenn du mehrere Runden implementierst: Was wird zwischen den Runden zurueckgesetzt? Der Geheimcode, die Versuchsliste, der Versuchszaehler -- alles. Was bleibt erhalten? Vielleicht der Punktestand. Mach diese Grenze explizit.

**Eingabe gross/klein**
Der Spieler tippt vielleicht `Rot` oder `ROT` statt `rot`. Entscheide ob dein Programm das akzeptiert -- und implementiere es dann konsequent. Beides ist eine gueltige Entscheidung.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Empfohlenes Vorgehen</span>
</div>

**Schritt 1 -- Die Bewertungsfunktion allein:**
Schreib nur die Funktion die aus Code und Versuch schwarze und weisse Treffer berechnet. Noch kein Spiel, kein Zufallscode, keine Eingabe. Nur diese eine Funktion -- und teste sie mit Handbeispielen:

```
bewerte([R, B, R, G], [R, R, B, Y]) -> schwarz=1, weiss=1
bewerte([R, R, B, G], [R, R, B, G]) -> schwarz=4, weiss=0
bewerte([R, B, G, Y], [Y, G, B, R]) -> schwarz=0, weiss=4
```

Erst wenn diese Funktion korrekt ist, weitermachen.

**Schritt 2 -- Das Grundspiel in der Konsole:**
Zufaelligen Code erzeugen, Eingabe einlesen, bewerten, ausgeben. Eine einfache Schleife, kein OOP noetig.

**Schritt 3 -- Struktur einfuehren:**
Wenn das Grundspiel laeuft, ueberlege ob sich eine Klasse lohnt. Vielleicht eine `Spiel`-Klasse die den Zustand haelt und Methoden wie `versuch_ausfuehren()` und `ist_beendet()` hat.

**Schritt 4 -- Ausbau:**
Statistiken, Schwierigkeitsgrade, GUI -- was immer dich interessiert.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Testen -- besonders hier wichtig</span>
</div>

Mastermind eignet sich hervorragend zum Testen mit pytest -- weil die Kernlogik reine Funktionen sind. Die Bewertungsfunktion hat keine Seiteneffekte, kein I/O, keinen Zustand: gleiche Eingabe, gleiche Ausgabe. Genau das was sich einfach testen laesst.

Schreib Tests fuer die Bewertungsfunktion bevor oder waehrend du sie implementierst. Das gibt dir Sicherheit wenn du sie spaeter anpasst.

Gute Testfaelle fuer die Bewertung:

```
# Alle richtig
bewerte([R,B,G,Y], [R,B,G,Y]) == (4, 0)

# Alle falsch
bewerte([R,R,R,R], [B,B,B,B]) == (0, 0)

# Alle richtige Farben, alle falsche Positionen
bewerte([R,B,G,Y], [B,G,Y,R]) == (0, 4)

# Wiederholungen im Code
bewerte([R,R,B,G], [R,R,R,R]) == (2, 0)

# Wiederholungen im Versuch
bewerte([R,B,G,Y], [R,R,R,R]) == (1, 0)
```

Wenn alle diese Tests bestehen, ist die Bewertungslogik korrekt.